# Day 9 — 효과크기, 신뢰구간, p-value: 세 숫자는 서로 다른 질문에 답한다

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-2/day-09.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-2/day-09.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 그룹당 6마리 실험에서 세 숫자를 모두 계산하기

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(11)
gc = rng.normal(loc=10.0, scale=1.0, size=6)    # 가상의 지상 대조군
flt = rng.normal(loc=11.5, scale=1.0, size=6)   # 가상의 비행군 (진짜 차이 1.5)

def cohens_d(a, b):
    """두 그룹 평균 차이를 '합동 표준편차' 단위로 나타낸다."""
    na, nb = len(a), len(b)
    pooled_sd = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / pooled_sd

diff = flt.mean() - gc.mean()
res = stats.ttest_ind(flt, gc, equal_var=False)       # Welch t-검정
ci = res.confidence_interval(confidence_level=0.95)

print(f"평균 차이 (FLT - GC): {diff:.2f}")
print(f"Cohen's d: {cohens_d(flt, gc):.2f}")
print(f"95% 신뢰구간: {ci.low:.2f} ~ {ci.high:.2f}")
print(f"p-value: {res.pvalue:.4f}")

### 예시 2 — 작은 p가 큰 효과를 뜻하지 않고, 큰 p가 효과 없음을 뜻하지 않는다

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(1)

def report(name, a, b):
    sd = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)   # 두 그룹 크기가 같을 때의 합동 SD
    d = (b.mean() - a.mean()) / sd
    p = stats.ttest_ind(b, a, equal_var=False).pvalue
    print(f"{name}: n={len(a)}/그룹, 평균 차이={b.mean() - a.mean():.2f}, d={d:.2f}, p={p:.4f}")

# A: 표본은 아주 크고, 진짜 차이는 아주 작다
report("A (큰 표본, 작은 효과)", rng.normal(10, 1, 5000), rng.normal(10.1, 1, 5000))
# B: 표본은 작고, 차이는 크다
report("B (작은 표본, 큰 효과)", np.array([8.9, 10.4, 9.6, 11.1]), np.array([9.8, 12.3, 10.2, 11.7]))

### 예시 3 — 진짜 차이가 전혀 없어도 20번에 1번은 p < 0.05

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)
n_experiments = 2000

# 두 그룹이 '완전히 같은' 모집단에서 나온 실험을 2000번 반복
a = rng.normal(10, 1, size=(n_experiments, 6))
b = rng.normal(10, 1, size=(n_experiments, 6))
p = stats.ttest_ind(a, b, axis=1, equal_var=False).pvalue

print(f"p < 0.05 인 실험: {(p < 0.05).sum()}개 / {n_experiments}개 ({(p < 0.05).mean():.1%})")

## 직접 해보기

1. 어제 만든 `notebooks/02_statistics_basics.ipynb`를 열고 맨 아래에 Markdown 셀로 `## Day 9 — 효과크기, 신뢰구간, p-value`를 추가합니다.
2. 예시 1을 실행해 네 숫자가 같은지 확인합니다.
3. 두 그룹을 그림으로 그립니다. 개별 점을 모두 찍고 그룹 평균을 빨간 가로선으로 표시합니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.DataFrame({"group": ["GC"] * 6 + ["FLT"] * 6, "expression": np.r_[gc, flt]})

fig, ax = plt.subplots(figsize=(4, 4))
sns.stripplot(data=df, x="group", y="expression", size=8, color="black", ax=ax)
for i, g in enumerate(["GC", "FLT"]):
    m = df.loc[df["group"] == g, "expression"].mean()
    ax.hlines(m, i - 0.25, i + 0.25, color="red")
ax.set_title(f"diff = {diff:.2f}, d = {cohens_d(flt, gc):.2f}")
plt.tight_layout()
plt.savefig("../figures/day09_two_groups.png", dpi=150)
plt.show()